# Muharaf transformer: prepare and measure the frozen model

**Goal:** run the Arabic handwriting transformer on the existing `public_1100/set0` validation lines before changing any weights. This is the transformer starting point; a later stage will fine-tune on the training lines. Do not use the test manifest during development.

1. In Colab, choose **Runtime → Change runtime type → GPU** (L4 or A100 if available).
2. Place `public_line_images.zip` and your `public_1100` manifest folder under **My Drive/Colab Notebooks/data-stuff**. The archive must contain `public/` with the original line PNG and TXT files. The manifest directory must contain `train.jsonl` and `validation.jsonl`, possibly inside a `set0` or `public_1100_set0` subfolder.
3. Run cells top to bottom. The first run copies the ZIP to Colab's temporary disk; outputs are saved to Drive. If the runtime resets, rerun the staging cells.

**Model:** [BushraAlmod03/ArTrOCR-HTR](https://huggingface.co/BushraAlmod03/ArTrOCR-HTR). Its card warns that diacritic recognition was not trained. We retain the references unchanged to measure that limitation. The checkpoint is downloaded by Colab; it is not included in this notebook.

In [ ]:
%pip -q install -U "transformers>=4.50,<6" accelerate jiwer safetensors

In [ ]:
import os, sys, torch, transformers
from google.colab import drive
drive.mount('/content/drive')
assert torch.cuda.is_available(), 'Change runtime type to GPU, then rerun.'
print('GPU:', torch.cuda.get_device_name(0))
print('Python:', sys.version.split()[0], 'PyTorch:', torch.__version__, 'Transformers:', transformers.__version__)

In [ ]:
from pathlib import Path
DATA_DIR = Path('/content/drive/MyDrive/Colab Notebooks/data-stuff')
assert DATA_DIR.is_dir(), f'Drive folder not found: {DATA_DIR}'
ZIP = DATA_DIR/'public_line_images.zip'
assert ZIP.is_file(), f'ZIP not found: {ZIP}'
candidate_dirs = [p.parent for p in DATA_DIR.rglob('validation.jsonl') if (p.parent/'train.jsonl').is_file()]
print('Archive:', ZIP)
print('Manifest directory candidates:', candidate_dirs)
assert len(candidate_dirs) == 1, 'Expected exactly one folder containing train.jsonl and validation.jsonl under data-stuff. Check the folder name/contents.'
MANIFESTS = candidate_dirs[0]
OUTPUT = DATA_DIR/'muharaf_transformer_runs'/'frozen_v1'
OUTPUT.mkdir(parents=True, exist_ok=True)
print('Outputs:', OUTPUT)

In [ ]:
import shutil, zipfile
LOCAL_ZIP = Path('/content/public_line_images.zip')
STAGING = Path('/content/muharaf_public')
if not LOCAL_ZIP.exists() or LOCAL_ZIP.stat().st_size != ZIP.stat().st_size:
    shutil.copyfile(ZIP, LOCAL_ZIP)
with zipfile.ZipFile(LOCAL_ZIP) as z:
    names = z.namelist()
    assert all(not Path(n).is_absolute() and '..' not in Path(n).parts for n in names), 'Archive contains unsafe names'
    assert any(n.startswith('public/') and n.endswith('.png') for n in names), 'Expected public/*.png inside ZIP'
    if not (STAGING/'public').exists():
        z.extractall(STAGING)
PUBLIC = STAGING/'public'
pngs = list(PUBLIC.rglob('*.png'))
print('Extracted PNGs:', len(pngs), 'at', PUBLIC)
assert len(pngs) == 24495, 'Expected 24,495 line images; check the ZIP and extraction.'
assert all(p.with_suffix('.txt').exists() for p in pngs), 'A line PNG is missing its TXT label.'

### Validate manifest fields and map lines to local files

The manifest may contain original paths from another computer. This cell maps by the PNG filename inside the ZIP, requires names to be unique, and checks that each manifest reference agrees with its unchanged TXT file. If your manifest uses different field names, the first printed record shows what to adapt.

In [ ]:
import json, collections
sample_record = json.loads((MANIFESTS/'train.jsonl').open(encoding='utf-8').readline())
print('Manifest field names:', list(sample_record))
print('One record (shortened):', {k:str(v)[:110] for k,v in sample_record.items()})

by_name = {}
for p in pngs:
    if p.name in by_name:
        raise ValueError(f'Duplicate image filename: {p.name}')
    by_name[p.name] = p

def pick(record, names):
    for k in names:
        if k in record and isinstance(record[k], str):
            return record[k]
    return None

def read_split(name):
    result = []
    with (MANIFESTS/f'{name}.jsonl').open(encoding='utf-8') as fh:
        for lineno, row in enumerate(fh, 1):
            r = json.loads(row)
            img = pick(r, ('image_path','image','image_file','png_path','line_image_path','line_image'))
            if not img:
                paths = [v for v in r.values() if isinstance(v,str) and v.lower().endswith('.png')]
                if len(paths) == 1: img = paths[0]
            assert img, f'No PNG field in {name}:{lineno}; inspect the printed record above'
            p = by_name.get(Path(img).name)
            assert p, f'Missing PNG for {name}:{lineno}: {img}'
            reference = p.with_suffix('.txt').read_text(encoding='utf-8')
            recorded = pick(r, ('text','transcription','ground_truth','gt','label_text','reference'))
            if recorded is not None:
                assert recorded == reference, f'Manifest text differs from original label for {p.name}'
            result.append({'image':p, 'reference':reference, 'source':r})
    assert len({x['image'] for x in result}) == len(result), f'Duplicate {name} image'
    return result

train, valid = read_split('train'), read_split('validation')
assert len(train) == 22092 and len(valid) == 1069, (len(train),len(valid))
assert not ({x['image'] for x in train} & {x['image'] for x in valid})
print('Training lines:',len(train),'Validation lines:',len(valid))
print('Example reference:',repr(valid[0]['reference'][:140]))

### Load transformer and verify it can encode every training label

The tokenizer check prevents silent dropping or changing of harakat. No checkpoint weights are updated in this stage. The author's card uses 512×102 images and positional interpolation. We reproduce that image sizing without its optional binarization or morphology.

In [ ]:
from transformers import VisionEncoderDecoderModel, TrOCRProcessor
MODEL_ID = 'BushraAlmod03/ArTrOCR-HTR'
processor = TrOCRProcessor.from_pretrained(MODEL_ID)
model = VisionEncoderDecoderModel.from_pretrained(MODEL_ID).to('cuda')
assert processor.image_processor.do_resize is False, 'Review processor resize behavior before proceeding.'
assert model.config.decoder_start_token_id is not None and model.config.pad_token_id is not None
special = processor.tokenizer.all_special_ids
bad = []
max_label_tokens = 0
for i, x in enumerate(train):
    ids = processor.tokenizer(x['reference'], add_special_tokens=False)['input_ids']
    max_label_tokens = max(max_label_tokens, len(ids))
    back = processor.tokenizer.decode(ids, skip_special_tokens=False, clean_up_tokenization_spaces=False)
    if back != x['reference'] or any(v in special for v in ids):
        bad.append((i,repr(x['reference'][:50]),repr(back[:50])))
print('Tokenizer non-roundtrip labels:',len(bad),'examples:',bad[:5])
assert not bad, 'Stop: tokenizer does not preserve every training label exactly.'
MAX_NEW_TOKENS = max_label_tokens + 8
decoder_limit = getattr(model.config.decoder, 'max_position_embeddings', None)
print('Longest training reference:',max_label_tokens,'tokens; generation allowance:',MAX_NEW_TOKENS,'decoder limit:',decoder_limit)
assert decoder_limit is None or MAX_NEW_TOKENS + 2 <= decoder_limit, 'A label exceeds the decoder position limit; review long lines before scoring.'

# The model card explicitly enables encoder position interpolation for generation.
if not hasattr(model.encoder, '_muharaf_original_forward'):
    model.encoder._muharaf_original_forward = model.encoder.forward
    def interpolated_forward(pixel_values, *args, **kwargs):
        kwargs['interpolate_pos_encoding'] = True
        return model.encoder._muharaf_original_forward(pixel_values, *args, **kwargs)
    model.encoder.forward = interpolated_forward
model.eval()
print('Model loaded; weights frozen for inference.')

In [ ]:
from PIL import Image
import numpy as np
def prepare_image(path):
    with Image.open(path) as src:
        img = src.convert('RGB')
    w,h = img.size
    target_w,target_h = 512,102
    new_w = max(1,round(w * target_h / h))
    img = img.resize((new_w,target_h),Image.Resampling.LANCZOS)
    if new_w > target_w:
        img = img.resize((target_w,target_h),Image.Resampling.LANCZOS)
        new_w = target_w
    canvas = Image.new('RGB',(target_w,target_h),(255,255,255))
    canvas.paste(img,(0,0))
    return canvas

def predict(item):
    im = prepare_image(item['image'])
    pixels = processor(images=im,return_tensors='pt').pixel_values.to('cuda')
    with torch.inference_mode():
        # One decoder-start token plus the same generated-token allowance.
        # Passing max_length alone avoids the checkpoint's max_length=20 warning.
        ids = model.generate(pixels,max_length=MAX_NEW_TOKENS+1,num_beams=1)
    return processor.batch_decode(ids,skip_special_tokens=True,clean_up_tokenization_spaces=False)[0]

print('Example:',valid[0]['image'].name)
print('Reference:',repr(valid[0]['reference']))
print('Prediction:',repr(predict(valid[0])))

### Frozen validation pass

This cell evaluates all 1,069 existing validation lines. It writes progress every 25 lines so a Colab disconnect need not erase the complete run. It will take longer than the single-line check. The output is development evidence because the supplied SFR checkpoint already used this validation group for model selection. The test set stays closed.

In [ ]:
import time
from jiwer import cer, wer
PREDICTIONS = OUTPUT/'validation_predictions.jsonl'
existing = {}
if PREDICTIONS.exists():
    for row in PREDICTIONS.open(encoding='utf-8'):
        r = json.loads(row)
        existing[r['image_name']] = r
print('Resuming with',len(existing),'predictions')
start = time.monotonic()
with PREDICTIONS.open('a',encoding='utf-8') as f:
    for idx,item in enumerate(valid,1):
        name = item['image'].name
        if name in existing:
            assert existing[name]['reference'] == item['reference'], f'Existing result reference differs: {name}'
            continue
        row = {'image_name':name,'reference':item['reference'],'prediction':predict(item)}
        f.write(json.dumps(row,ensure_ascii=False)+'\n')
        f.flush()
        existing[name] = row
        if idx % 25 == 0:
            print(f'{idx}/{len(valid)} processed; elapsed {time.monotonic()-start:.1f}s')
assert len(existing) == len(valid)
ordered = [existing[x['image'].name] for x in valid]
refs = [r['reference'] for r in ordered]
hyps = [r['prediction'] for r in ordered]
marks = set('ًٌٍَُِّْ')
remove_marks = lambda s: ''.join(ch for ch in s if ch not in marks)
results = {'model':MODEL_ID,'split':'public_1100/set0 validation','lines':len(valid),
           'raw_cer':cer(refs,hyps),'wer':wer(refs,hyps),
           'cer_without_basic_harakat':cer(list(map(remove_marks,refs)),list(map(remove_marks,hyps))),
           'reference_basic_harakat':sum(ch in marks for s in refs for ch in s),
           'predicted_basic_harakat':sum(ch in marks for s in hyps for ch in s),
           'torch':torch.__version__,'transformers':transformers.__version__}
(OUTPUT/'validation_metrics.json').write_text(json.dumps(results,ensure_ascii=False,indent=2),encoding='utf-8')
print(json.dumps(results,ensure_ascii=False,indent=2))
print('Saved:',OUTPUT)
print('Next: review errors and launch a separate fine-tuning stage on train only.')